# 08: Visual Comparison of Pointcloud Labeling Methods

This notebook provides a modular and reproducible framework to visually compare different strategies for projecting 2D segmentation masks onto a 3D pointcloud, using Project Aria data.

### In this notebook, we will:
1. Data loading and setup
2. Mask extraction utilities (SAM2)
3. Baseline projection: single frame, single camera
4. Accumulated projection: different frames combination
5. Multi-camera projection: different camera combination
6. Final comparison: best of 4. vs best of 5.
7. Majority Voting System
8. New final comparison

As a sample, we will use `kettle_and_forklift_recording.vrs` located in the local `data/raw/kettle_and_forklift/` directory and its corresponding segmentation masks located in `data/outputs/segmentation/kettle_and_forklift/sam2/masks` .


## 8.1 Data Loading and Setup

In this section, we load the 3D pointcloud and camera trajectory data produced by MPS

In [1]:
# Import required libraries and load pointcloud and trajectory data
import os
import pandas as pd
from aria_pylib import find_points_file, find_trajectory_file

# Define the directory containing MPS outputs
MPS_DIR = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'mps_kettle_and_forklift_recording_vrs', 'slam')

# Load the pointcloud file
points_path = find_points_file(MPS_DIR)
print(f"Loading pointcloud from: {points_path}")
points_df = pd.read_csv(points_path)

# Load trajectory file
trajectory_path = find_trajectory_file(MPS_DIR)
print(f"Loading trajectory from: {trajectory_path}")
trajectory_df = pd.read_csv(trajectory_path)
print(f"Loaded {len(trajectory_df)} poses.")

Loading pointcloud from: ..\data\raw\kettle_and_forklift\mps_kettle_and_forklift_recording_vrs\slam\semidense_points.csv.gz
Loading trajectory from: ..\data\raw\kettle_and_forklift\mps_kettle_and_forklift_recording_vrs\slam\closed_loop_trajectory.csv
Loaded 50626 poses.


## 8.2 Mask Extraction Utilities

This section provides utility functions to extract 2D segmentation masks from the dataset, given a specific timestamp and camera. These functions will be used to retrieve the masks needed for each projection experiment.

In [2]:
# Utility function to load a segmentation mask for a given frame and camera
import cv2

def load_mask(masks_dir, frame_idx, camera_name="camera-rgb"):
    """
    Loads a 2D segmentation mask for the specified frame and camera.
    """
    mask_filename = f"{frame_idx:06d}.png"
    mask_path = os.path.join(masks_dir, camera_name, mask_filename) if os.path.isdir(os.path.join(masks_dir, camera_name)) else os.path.join(masks_dir, mask_filename)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"Could not load mask at {mask_path}. Check if the file exists!")
    print(f"Loaded mask from {mask_path}\nShape: {mask.shape}")
    return mask

# example
masks_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'sam2', 'masks')
frame_idx = 50
mask = load_mask(masks_dir, frame_idx)

Loaded mask from ..\data\outputs\segmentation\kettle_and_forklift\sam2\masks\000050.png
Shape: (1408, 1408)


## 8.3.1 Baseline Projection: Single Frame, Single Camera

In this section, we implement the baseline projection method: projecting the 3D pointcloud onto a single 2D segmentation mask (from the RGB camera) at a specific frame. The resulting labeled pointcloud will be used as a reference for comparison with more advanced strategies.
We will use the frame 50 (video at 10 fps) as an example

In [3]:
import os
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.spatial.transform import Rotation as R
from projectaria_tools.core import data_provider

# Setup paths and parameters
vrs_path = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'kettle_and_forklift_recording.vrs')
masks_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'sam2', 'masks')
baseline_frame_idx = 50 

# Load Calibration
provider = data_provider.create_vrs_data_provider(vrs_path)
device_calib = provider.get_device_calibration()
rgb_calib = device_calib.get_camera_calib("camera-rgb")
T_Device_Camera = rgb_calib.get_transform_device_camera()

# Helper Functions
def load_mask(masks_dir, frame_idx):
    """Loads a 2D segmentation mask from disk."""
    mask_path = os.path.join(masks_dir, f"{frame_idx:06d}.png")
    return cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

def get_pose_from_vrs_timestamp(provider, trajectory_df, frame_idx, seg_fps=10, vrs_fps=30, stream_name="camera-rgb"):
    """Extracts exact hardware timestamp and finds the closest trajectory pose."""
    # Scale index from segmentation fps to native VRS fps
    vrs_frame_idx = int(frame_idx * (vrs_fps / seg_fps))
    
    stream_id = provider.get_stream_id_from_label(stream_name)
    image_data = provider.get_image_data_by_index(stream_id, vrs_frame_idx)
    capture_time_ns = image_data[1].capture_timestamp_ns
    
    # Match nanoseconds with the correct trajectory column
    if 'tracking_timestamp_us' in trajectory_df.columns:
        trajectory_times_ns = trajectory_df['tracking_timestamp_us'].to_numpy() * 1000
    elif 'timestamp_ns' in trajectory_df.columns:
        trajectory_times_ns = trajectory_df['timestamp_ns'].to_numpy()
    elif 'utc_timestamp_ns' in trajectory_df.columns:
        trajectory_times_ns = trajectory_df['utc_timestamp_ns'].to_numpy()
    else:
        raise KeyError(f"Timestamp column not found. Available: {trajectory_df.columns.tolist()}")

    closest_traj_idx = (np.abs(trajectory_times_ns - capture_time_ns)).argmin()
    return trajectory_df.iloc[closest_traj_idx]


def project_world_to_rgb_image(point_world, pose_row, rgb_calib, T_Device_Camera):
    """Projects a 3D point into 2D pixel coordinates."""
    quat = [pose_row['qx_world_device'], pose_row['qy_world_device'], pose_row['qz_world_device'], pose_row['qw_world_device']]
    rot_world_device = R.from_quat(quat).as_matrix()
    t_world_device = np.array([pose_row['tx_world_device'], pose_row['ty_world_device'], pose_row['tz_world_device']])
    
    point_device = rot_world_device.T @ (point_world - t_world_device)
    point_cam = T_Device_Camera.inverse() @ point_device
    return rgb_calib.project(point_cam)


def label_pointcloud_at_pose(points_df, pose_row, rgb_calib, T_Device_Camera, mask):
    """Assigns semantic labels to 3D points based on 2D mask projection."""
    labels, px, py, pz = [], [], [], []
    for _, row in tqdm(points_df.iterrows(), total=len(points_df), desc=f"Projecting points"):
        point_3d = np.array([row['px_world'], row['py_world'], row['pz_world']])
        pixel = project_world_to_rgb_image(point_3d, pose_row, rgb_calib, T_Device_Camera)
        label = 0
        if pixel is not None:
            u, v = int(round(pixel[0])), int(round(pixel[1]))
            if 0 <= v < mask.shape[0] and 0 <= u < mask.shape[1]:
                label = mask[v, u]
        labels.append(label)
        px.append(point_3d[0]); py.append(point_3d[1]); pz.append(point_3d[2])
    return pd.DataFrame({'px_world': px, 'py_world': py, 'pz_world': pz, 'semantic_label': labels})


print(f"Aligning mask for frame {baseline_frame_idx}...")
baseline_mask = load_mask(masks_dir, baseline_frame_idx)
baseline_pose = get_pose_from_vrs_timestamp(provider, trajectory_df, baseline_frame_idx)

print(f"Trajectory perfectly matched at index: {baseline_pose.name}")
labeled_baseline_df = label_pointcloud_at_pose(points_df, baseline_pose, rgb_calib, T_Device_Camera, baseline_mask)
print(f"Unique labels in 3D: {np.unique(labeled_baseline_df['semantic_label'])}")

Aligning mask for frame 50...
Trajectory perfectly matched at index: 4033


Projecting points: 100%|██████████| 239524/239524 [00:17<00:00, 13990.31it/s]


Unique labels in 3D: [0 1 2]


### 8.3.2 Visualize Baseline Projection in 3D
To verify the alignment, we visualize the labeled points in 3D using Rerun. We filter out the background (label 0) to exclusively inspect the points hit by the segmentation mask, along with the exact camera pose at the moment of capture.

In [4]:
import rerun as rr

# Filter out background to isolate target objects
valid_mask = labeled_baseline_df['semantic_label'] > 0
target_points = labeled_baseline_df[valid_mask]

points_xyz = target_points[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
labels = target_points['semantic_label'].to_numpy()

# 1 = Red/Forklift, 2 = Cyan/Kettle
label_to_color = {1: [255, 0, 0], 2: [0, 255, 255]}
colors = np.array([label_to_color.get(int(l), [0, 255, 0]) for l in labels], dtype=np.uint8)

# Center the world around the trajectory for smooth navigation
traj_xyz = trajectory_df[['tx_world_device', 'ty_world_device', 'tz_world_device']].to_numpy(dtype=np.float32)
center = np.mean(traj_xyz, axis=0, keepdims=True)

points_xyz_centered = points_xyz - center
traj_xyz_centered = traj_xyz - center

# Camera position at capture time
camera_xyz = np.array([baseline_pose['tx_world_device'], baseline_pose['ty_world_device'], baseline_pose['tz_world_device']], dtype=np.float32)
camera_centered = camera_xyz - center

# Launch Rerun visualization
rr.init("Aria_Baseline_Frame_50", spawn=True)
rr.log("pointcloud/target_labels", rr.Points3D(points_xyz_centered, colors=colors, radii=0.0005))
rr.log("camera/trajectory", rr.LineStrips3D([traj_xyz_centered], colors=[[0,255,0]]))
rr.log("camera/pose_frame_50", rr.Points3D([camera_centered], colors=[[255,255,0]], radii=0.003))

print(f"Sent {len(points_xyz)} target points to Rerun viewer.")

Sent 41108 target points to Rerun viewer.


### 8.3.3 Save Baseline Pointcloud
We will now save the entire labeled pointcloud as a `.ply` file for external analysis

In [5]:
import os
import numpy as np
from plyfile import PlyData, PlyElement

print(f"Saving the entire labeled pointcloud to disk...")
pcd_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'pointclouds', 'multi_frame_comparison')
os.makedirs(pcd_dir, exist_ok=True)

# Extract ALL points and labels
all_xyz = labeled_baseline_df[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
all_labels = labeled_baseline_df['semantic_label'].to_numpy()

save_label_to_color = {0: [180, 180, 180], 1: [255, 0, 0], 2: [0, 255, 255]}
all_colors = np.array([save_label_to_color.get(int(l), [180, 180, 180]) for l in all_labels], dtype=np.uint8)

# Compose structured array for ply format
vertex = np.empty(all_xyz.shape[0], dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'), ('red', 'u1'), ('green', 'u1'), ('blue', 'u1'), ('label', 'u1')])
vertex['x'] = all_xyz[:, 0]
vertex['y'] = all_xyz[:, 1]
vertex['z'] = all_xyz[:, 2]
vertex['red'] = all_colors[:, 0]
vertex['green'] = all_colors[:, 1]
vertex['blue'] = all_colors[:, 2]
vertex['label'] = all_labels.astype(np.uint8)

# Write to disk only if the file does not already exist
out_path = os.path.join(pcd_dir, f"pointcloud_frame_{baseline_frame_idx}.ply")
if os.path.exists(out_path):
    print(f"Skipping existing file: {out_path}")
else:
    ply = PlyData([PlyElement.describe(vertex, 'vertex')], text=True)
    ply.write(out_path)
    print(f"Full pointcloud saved successfully at: {out_path}")

Saving the entire labeled pointcloud to disk...
Skipping existing file: ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\multi_frame_comparison\pointcloud_frame_50.ply


## 8.4 Multi-Frame Projection Comparison

In this section, we compare the effect of combining different sets of frames for the projection. We will accumulate the labels from multiple frames and analyze how the coverage and quality of the labeled pointcloud change as we add more frames.

We will test the following frame sets:
1) Only frame 50 (done previously in the baseline)
2) Frames 50 + 100 + 250
3) Frames 50 + 100 + 250 + 350 + 480
4) Frames 50 + 100 + 250 + 350 + 480 + 150 + 200

All the pointclouds will be saved as .ply files for external analysis but only the final comparison will be visualized in Rerun for qualitative analysis.

### 8.4.1 Labeling Accumulation from Multiple Frames
We will define an accumulation function that iterates over a given set of frames. For each frame, it fetches the exact hardware pose, projects the points, and updates the global labels (overwriting background points with newly discovered semantic labels).

In [6]:
import matplotlib.pyplot as plt
import numpy as np

# Single-frame mask viewer for the comparison frames
comparison_frames = [50, 100, 250, 350, 480, 150, 200]
color_map = {0: (180, 180, 180), 1: (255, 0, 0), 2: (0, 255, 255)}

def show_mask_frame(frame_idx):
    mask = load_mask(masks_dir, frame_idx)
    rgb_mask = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
    for lbl, col in color_map.items():
        rgb_mask[mask == lbl] = col
    plt.figure(figsize=(10,6))
    plt.imshow(rgb_mask)
    plt.title(f"Mask frame {frame_idx}")
    plt.axis('off')
    plt.show()

# Try interactive widget (dropdown) if available, fallback to manual call
try:
    from ipywidgets import interact
    display(interact(show_mask_frame, frame_idx=comparison_frames))
except Exception:
    print('ipywidgets not available, showing masks sequentially.')


interactive(children=(Dropdown(description='frame_idx', options=(50, 100, 250, 350, 480, 150, 200), value=50),…

<function __main__.show_mask_frame(frame_idx)>

In [7]:
from collections import OrderedDict
import numpy as np
import pandas as pd

# Multi-Frame Accumulation Helper
def accumulate_labels_multi_frame(points_df, trajectory_df, frame_indices, rgb_calib, T_Device_Camera, masks_dir, provider, mask_cache=None):
    """
    Projects and accumulates labels from multiple frames onto the 3D pointcloud.
    Uses exact hardware pose lookup for each frame to guarantee perfect alignment.
    """
    if mask_cache is None:
        mask_cache = {}
        
    # Start with all points labeled as 0 (background)
    labels_accum = np.zeros(len(points_df), dtype=int)
    
    for frame_idx in frame_indices:
        # Load mask (using cache to save I/O time)
        if frame_idx not in mask_cache:
            mask_cache[frame_idx] = load_mask(masks_dir, frame_idx)
        mask = mask_cache[frame_idx]
        
        # Get the exact hardware pose for this frame
        pose_row = get_pose_from_vrs_timestamp(provider, trajectory_df, frame_idx)
        
        # Project and label
        labeled_tmp = label_pointcloud_at_pose(points_df, pose_row, rgb_calib, T_Device_Camera, mask)
        
        # Accumulate labels: overwrite current 0s with new valid labels (>0)
        labels_accum = np.where(labeled_tmp['semantic_label'] > 0, labeled_tmp['semantic_label'], labels_accum)
        
    labeled_df = points_df.copy()
    labeled_df['semantic_label'] = labels_accum
    return labeled_df

# Define Frame Sets
frame_sets = [
    [50, 100, 250],
    [50, 100, 250, 350, 480],
    [50, 100, 250, 350, 480, 150, 200]
]

# Pre-load masks
all_frames = list(set([f for subset in frame_sets for f in subset]))
mask_cache = OrderedDict()
print("Pre-loading SAM2 masks to optimize execution...")
for f in all_frames:
    mask_cache[f] = load_mask(masks_dir, f)

# Execute Accumulation
labeled_pointclouds = {}

for frames in frame_sets:
    dict_key = f"frames_{'_'.join(map(str, frames))}"
    print(f"\nAccumulating projection for {len(frames)} frames: {frames}")
    
    labeled_pc = accumulate_labels_multi_frame(
        points_df, trajectory_df, frames, rgb_calib, T_Device_Camera, masks_dir, provider, mask_cache=mask_cache
    )
    
    labeled_pointclouds[dict_key] = labeled_pc
    print(f"Completed accumulation for {dict_key}")

Pre-loading SAM2 masks to optimize execution...

Accumulating projection for 3 frames: [50, 100, 250]


Projecting points:  35%|███▌      | 84094/239524 [00:06<00:11, 13411.24it/s]


KeyboardInterrupt: 

### 8.4.2 Save Multi-Frame Results as .ply
We export the accumulated pointclouds for each frame set. These files contain the full geometry and the updated semantic colors, ready for quantitative evaluation.

In [8]:
import os
import numpy as np
from plyfile import PlyData, PlyElement

# Output directory for the accumulated pointclouds
pcd_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'pointclouds', 'multi_frame_comparison')
os.makedirs(pcd_dir, exist_ok=True)

# Color mapping (0 = Grey, 1 = Red, 2 = Cyan)
save_label_to_color = {0: [180, 180, 180], 1: [255, 0, 0], 2: [0, 255, 255]}

for key, labeled_pc in labeled_pointclouds.items():
    print(f"Exporting pointcloud: {key} ...")

    all_xyz = labeled_pc[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
    all_labels = labeled_pc['semantic_label'].to_numpy()
    all_colors = np.array([save_label_to_color.get(int(l), [180, 180, 180]) for l in all_labels], dtype=np.uint8)

    # Compose structured array for the .ply format
    vertex = np.empty(all_xyz.shape[0], dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'), ('red', 'u1'), ('green', 'u1'), ('blue', 'u1'), ('label', 'u1')])
    vertex['x'] = all_xyz[:, 0]
    vertex['y'] = all_xyz[:, 1]
    vertex['z'] = all_xyz[:, 2]
    vertex['red'] = all_colors[:, 0]
    vertex['green'] = all_colors[:, 1]
    vertex['blue'] = all_colors[:, 2]
    vertex['label'] = all_labels.astype(np.uint8)

    # Write to disk only if the file does not already exist
    out_path = os.path.join(pcd_dir, f"pointcloud_{key}.ply")
    if os.path.exists(out_path):
        print(f"Skipping existing file: {out_path}")
        continue

    ply = PlyData([PlyElement.describe(vertex, 'vertex')], text=True)
    ply.write(out_path)
    print(f"Saved: {out_path}")

### 8.4.3 Visual Comparison: 1 Frame vs 7 Frames
We use Rerun to qualitatively compare the baseline projection (1 frame) against our maximum accumulation set (7 frames). This highlights how multi-frame projection resolves occlusion issues and fills in the 3D geometry of the target objects.

In [9]:
import rerun as rr
import numpy as np

# Get Baseline (1 Frame) and Accumulated (7 Frames) 
baseline_df = labeled_baseline_df
accum_df = labeled_pointclouds['frames_50_100_250_350_480_150_200']

# Filter Valid Points Only 
baseline_valid = baseline_df[baseline_df['semantic_label'] > 0]
accum_valid = accum_df[accum_df['semantic_label'] > 0]

baseline_xyz = baseline_valid[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
baseline_labels = baseline_valid['semantic_label'].to_numpy()

accum_xyz = accum_valid[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
accum_labels = accum_valid['semantic_label'].to_numpy()

# Color Mapping
vis_label_to_color = {1: [255, 0, 0], 2: [0, 255, 255]}
baseline_colors = np.array([vis_label_to_color.get(int(l), [0, 255, 0]) for l in baseline_labels], dtype=np.uint8)
accum_colors = np.array([vis_label_to_color.get(int(l), [0, 255, 0]) for l in accum_labels], dtype=np.uint8)

# Center World 
traj_xyz = trajectory_df[['tx_world_device', 'ty_world_device', 'tz_world_device']].to_numpy(dtype=np.float32)
center = np.mean(traj_xyz, axis=0, keepdims=True)

# Rerun Visualization
rr.init("Aria_MultiFrame_Comparison", spawn=True)

rr.log("world/trajectory", rr.LineStrips3D([traj_xyz - center], colors=[[0,255,0]]))

# Log Baseline 
rr.log("world/pointcloud/1_frame_baseline", rr.Points3D(baseline_xyz - center, colors=baseline_colors, radii=0.0005))

# Log Accumulated 
rr.log("world/pointcloud/7_frames_accumulated", rr.Points3D(accum_xyz - center, colors=accum_colors, radii=0.0005))

print(f"Sent comparison to Rerun:")
print(f"- Baseline (1 Frame): {len(baseline_xyz)} points")
print(f"- Accumulated (7 Frames): {len(accum_xyz)} points")
print("Toggle visibility in the Rerun UI to compare the coverage.")

KeyError: 'frames_50_100_250_350_480_150_200'

### Multi frame Comparison conlusion
Based on the visual and quantitative comparison, the pointcloud obtained by accumulating labels from 3 frames (50, 100, 250) provides the best trade-off between correct semantic labeling and the presence of outliers. This configuration maximizes the number of correctly labeled points on the target objects while minimizing the inclusion of spurious or misprojected points. Adding more frames increases coverage but also introduces more outliers, so the 3-frame result is recommended for downstream tasks and visualization and will be used for the final comparison with multi-camera projection.

## 8.5 Camera-Specific Projection Comparison

In this section, each camera produces its own pointcloud from the same synchronized frame used in the baseline (frame 50). The goal is to build a camera-specific visible subset of the 3D cloud, so that the comparison reflects how each camera sees the scene from that exact shot.

### 8.5.1 Camera Visibility Model

For each camera, we transform the 3D points into that camera frame, keep only points with positive depth, keep only points that project inside the image, and keep only the closest point per pixel.

This version evaluates only frame 50. That keeps the comparison aligned with the baseline.

In [10]:
from scipy.spatial.transform import Rotation as R

# Cache the point coordinates once
points_world_all = points_df[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float64)
point_ids_all = points_df['point_id'].to_numpy(dtype=np.int64) if 'point_id' in points_df.columns else np.arange(len(points_df), dtype=np.int64)
if 'point_id' not in points_df.columns:
    points_df = points_df.copy()
    points_df['point_id'] = point_ids_all


def get_camera_calibration(camera_name):
    """Return the camera calibration and the device-to-camera transform for one camera."""
    camera_calib = device_calib.get_camera_calib(camera_name)
    T_Device_Camera = camera_calib.get_transform_device_camera()
    return camera_calib, T_Device_Camera


def get_pose_for_camera_frame(frame_idx, camera_name, pose_cache=None):
    """Return the synchronized pose for one camera/frame pair, using an optional cache."""
    cache_key = (camera_name, int(frame_idx))
    if pose_cache is not None and cache_key in pose_cache:
        return pose_cache[cache_key]

    pose_row = get_pose_from_vrs_timestamp(provider, trajectory_df, int(frame_idx), stream_name=camera_name)
    if pose_cache is not None:
        pose_cache[cache_key] = pose_row
    return pose_row


def project_world_to_camera(point_world, pose_row, T_Device_Camera):
    """Transform a world-space point into camera coordinates."""
    quat = [pose_row['qx_world_device'], pose_row['qy_world_device'], pose_row['qz_world_device'], pose_row['qw_world_device']]
    rot_world_device = R.from_quat(quat).as_matrix()
    t_world_device = np.array([pose_row['tx_world_device'], pose_row['ty_world_device'], pose_row['tz_world_device']], dtype=np.float64)
    point_device = rot_world_device.T @ (point_world - t_world_device)
    point_camera = T_Device_Camera.inverse() @ point_device
    return np.asarray(point_camera, dtype=np.float64).reshape(-1)


def build_visible_cloud_for_camera_frame(frame_idx, camera_name, pose_cache=None):
    """Build a camera-specific visible pointcloud for one synchronized frame."""
    if pose_cache is None:
        pose_cache = {}

    camera_calib, T_Device_Camera = get_camera_calibration(camera_name)
    pose_row = get_pose_for_camera_frame(frame_idx, camera_name, pose_cache=pose_cache)
    image_size = camera_calib.get_image_size()
    image_width = int(image_size[0])
    image_height = int(image_size[1])

    quat = [pose_row['qx_world_device'], pose_row['qy_world_device'], pose_row['qz_world_device'], pose_row['qw_world_device']]
    rot_world_device = R.from_quat(quat).as_matrix()
    t_world_device = np.array([pose_row['tx_world_device'], pose_row['ty_world_device'], pose_row['tz_world_device']], dtype=np.float64)
    T_Camera_Device = T_Device_Camera.inverse()

    depth_buffer = {}
    projection_buffer = {}

    for point_id, point_world in zip(point_ids_all, points_world_all):
        point_device = rot_world_device.T @ (point_world - t_world_device)
        point_camera = np.asarray(T_Camera_Device @ point_device, dtype=np.float64).reshape(-1)

        if point_camera[2] <= 0:
            continue

        pixel = camera_calib.project(point_camera)
        if pixel is None:
            continue

        u = int(round(float(pixel[0])))
        v = int(round(float(pixel[1])))
        if not (0 <= u < image_width and 0 <= v < image_height):
            continue

        depth = float(point_camera[2])
        pixel_key = (u, v)
        if pixel_key not in depth_buffer or depth < depth_buffer[pixel_key]:
            depth_buffer[pixel_key] = depth
            projection_buffer[pixel_key] = {
                'point_id': int(point_id),
                'px_world': float(point_world[0]),
                'py_world': float(point_world[1]),
                'pz_world': float(point_world[2]),
                'pixel_u': u,
                'pixel_v': v,
                'source_camera': camera_name,
                'source_frame_idx': int(frame_idx),
            }

    return pd.DataFrame(projection_buffer.values())


def build_visible_cloud_for_camera(frame_idx, camera_name, pose_cache=None):
    """Return the visible subset for one camera at a single frame."""
    return build_visible_cloud_for_camera_frame(frame_idx, camera_name, pose_cache=pose_cache)

### 8.5.2 Build and Save the Clouds

This section will compute one visible cloud per camera at frame 50, then reuses those camera-specific clouds to build the five tests

In [11]:
def build_pointcloud_for_camera_set(camera_names, precomputed_visible_clouds):
    """Merge the visible camera-specific clouds into one pointcloud."""
    merged_df = pd.concat([precomputed_visible_clouds[camera_name] for camera_name in camera_names], ignore_index=True)
    if merged_df.empty:
        return pd.DataFrame(columns=['point_id', 'px_world', 'py_world', 'pz_world', 'pixel_u', 'pixel_v', 'source_camera', 'source_frame_idx'])

    merged_df = merged_df.sort_values(['point_id', 'source_camera', 'source_frame_idx']).drop_duplicates(subset=['point_id'], keep='first')
    return merged_df.reset_index(drop=True)


def save_pointcloud_if_missing(pointcloud_df, out_path):
    """Write a PLY file only when it does not already exist."""
    if pointcloud_df.empty:
        print(f"Skipping empty pointcloud: {out_path}")
        return False

    if os.path.exists(out_path):
        print(f"Skipping existing file: {out_path}")
        return False

    camera_palette = {
        'camera-rgb': [255, 0, 0],
        'camera-slam-left': [0, 255, 0],
        'camera-slam-right': [0, 128, 255],
    }

    # Extract XYZ and assign colors based on source camera
    points_xyz = pointcloud_df[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float32)
    source_cameras = pointcloud_df['source_camera'].to_numpy()
    colors = np.array([camera_palette.get(str(camera), [180, 180, 180]) for camera in source_cameras], dtype=np.uint8)

    # Compose structured array for ply format
    vertex = np.empty(
        points_xyz.shape[0],
        dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'), ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')],
    )
    vertex['x'] = points_xyz[:, 0]
    vertex['y'] = points_xyz[:, 1]
    vertex['z'] = points_xyz[:, 2]
    vertex['red'] = colors[:, 0]
    vertex['green'] = colors[:, 1]
    vertex['blue'] = colors[:, 2]

    ply = PlyData([PlyElement.describe(vertex, 'vertex')], text=True)
    ply.write(out_path)
    print(f"Saved: {out_path}")
    return True


def summarize_camera_pointcloud(pointcloud_df):
    """Return a compact summary for a camera pointcloud."""
    if pointcloud_df.empty:
        return {
            'total_points': 0,
            'rgb_points': 0,
            'left_points': 0,
            'right_points': 0,
        }

    source_camera = pointcloud_df['source_camera']
    return {
        'total_points': int(len(pointcloud_df)),
        'rgb_points': int((source_camera == 'camera-rgb').sum()),
        'left_points': int((source_camera == 'camera-slam-left').sum()),
        'right_points': int((source_camera == 'camera-slam-right').sum()),
    }


def camera_pose_summary(pose_row, camera_name, source_frame_idx):
    """Summarize the world-space camera pose for CloudCompare reference."""
    quat = [pose_row['qx_world_device'], pose_row['qy_world_device'], pose_row['qz_world_device'], pose_row['qw_world_device']]
    rot_world_device = R.from_quat(quat).as_matrix()
    camera_position = np.array([pose_row['tx_world_device'], pose_row['ty_world_device'], pose_row['tz_world_device']], dtype=np.float64)
    forward_device = np.array([0.0, 0.0, 1.0], dtype=np.float64)
    up_device = np.array([0.0, -1.0, 0.0], dtype=np.float64)
    forward_world = rot_world_device @ forward_device
    up_world = rot_world_device @ up_device
    return {
        'camera_name': camera_name,
        'source_frame_idx': int(source_frame_idx),
        'trajectory_index': int(pose_row.name),
        'position_x': float(camera_position[0]),
        'position_y': float(camera_position[1]),
        'position_z': float(camera_position[2]),
        'forward_x': float(forward_world[0]),
        'forward_y': float(forward_world[1]),
        'forward_z': float(forward_world[2]),
        'up_x': float(up_world[0]),
        'up_y': float(up_world[1]),
        'up_z': float(up_world[2]),
    }

In [12]:
comparison_frame_idx = 50
comparison_cameras = ["camera-rgb", "camera-slam-left", "camera-slam-right"]
camera_tests = OrderedDict([
    ("test_1_rgb", ["camera-rgb"]),
    ("test_1_left", ["camera-slam-left"]),
    ("test_1_right", ["camera-slam-right"]),
    ("test_2_laterals", ["camera-slam-left", "camera-slam-right"]),
    ("test_3_all_cameras", ["camera-rgb", "camera-slam-left", "camera-slam-right"]),
])

# Output directory for the multi-camera comparison pointclouds.
mc_pcd_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'pointclouds', 'multi_camera_comparison')
os.makedirs(mc_pcd_dir, exist_ok=True)

precomputed_visible_clouds = {}
pose_cache_mc = {}
pose_summary_rows = []

for camera_name in comparison_cameras:
    print(f"Precomputing visible points for {camera_name} at frame {comparison_frame_idx}")
    pose_row = get_pose_for_camera_frame(comparison_frame_idx, camera_name, pose_cache=pose_cache_mc)
    pose_summary_rows.append(camera_pose_summary(pose_row, camera_name, comparison_frame_idx))
    precomputed_visible_clouds[camera_name] = build_visible_cloud_for_camera(
        comparison_frame_idx,
        camera_name,
        pose_cache=pose_cache_mc,
    )

pose_summary_df = pd.DataFrame(pose_summary_rows)
display(pose_summary_df[['camera_name', 'source_frame_idx', 'trajectory_index', 'position_x', 'position_y', 'position_z', 'forward_x', 'forward_y', 'forward_z']])

# Build and save all comparison pointclouds.
multi_camera_pointclouds = {}
comparison_summary_rows = []

for test_name, camera_names in camera_tests.items():
    print(f"Building pointcloud for {test_name}: {camera_names}")
    pointcloud_df = build_pointcloud_for_camera_set(camera_names, precomputed_visible_clouds)
    multi_camera_pointclouds[test_name] = pointcloud_df

    summary_row = summarize_camera_pointcloud(pointcloud_df)
    summary_row['test_name'] = test_name
    summary_row['cameras'] = ', '.join(camera_names)
    summary_row['frame_idx'] = comparison_frame_idx
    comparison_summary_rows.append(summary_row)

    out_path = os.path.join(mc_pcd_dir, f"pointcloud_{test_name}.ply")
    save_pointcloud_if_missing(pointcloud_df, out_path)
    print(f"Finished {test_name} with {len(pointcloud_df)} points.")

comparison_summary_df = pd.DataFrame(comparison_summary_rows)
display(comparison_summary_df[['test_name', 'cameras', 'frame_idx', 'total_points', 'rgb_points', 'left_points', 'right_points']])

Precomputing visible points for camera-rgb at frame 50
Precomputing visible points for camera-slam-left at frame 50
Precomputing visible points for camera-slam-right at frame 50


,camera_name,source_frame_idx,trajectory_index,position_x,position_y,position_z,forward_x,forward_y,forward_z
0,camera-rgb,50,4033,-0.002216,0.499352,0.019901,0.744316,-0.539806,-0.393198
1,camera-slam-left,50,4033,-0.002216,0.499352,0.019901,0.744316,-0.539806,-0.393198
2,camera-slam-right,50,4033,-0.002216,0.499352,0.019901,0.744316,-0.539806,-0.393198


Building pointcloud for test_1_rgb: ['camera-rgb']
Skipping existing file: ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\multi_camera_comparison\pointcloud_test_1_rgb.ply
Finished test_1_rgb with 67380 points.
Building pointcloud for test_1_left: ['camera-slam-left']
Skipping existing file: ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\multi_camera_comparison\pointcloud_test_1_left.ply
Finished test_1_left with 64757 points.
Building pointcloud for test_1_right: ['camera-slam-right']
Skipping existing file: ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\multi_camera_comparison\pointcloud_test_1_right.ply
Finished test_1_right with 39721 points.
Building pointcloud for test_2_laterals: ['camera-slam-left', 'camera-slam-right']
Skipping existing file: ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\multi_camera_comparison\pointcloud_test_2_laterals.ply
Finished test_2_laterals with 92274 points.
Building pointcloud for test_3_all_c

,test_name,cameras,frame_idx,total_points,rgb_points,left_points,right_points
0,test_1_rgb,camera-rgb,50,67380,67380,0,0
1,test_1_left,camera-slam-left,50,64757,0,64757,0
2,test_1_right,camera-slam-right,50,39721,0,0,39721
3,test_2_laterals,"camera-slam-left, camera-slam-right",50,92274,0,64757,27517
4,test_3_all_cameras,"camera-rgb, camera-slam-left, camera-slam-right",50,118191,67380,36732,14079


### Conclusion on Camera-Specific Projection

As expected, the best pointcloud was generating all the cameras together, which is the most permissive configuration. However, the second best configuration is using only the RGB camera, which is a promising result for applications that want to focus on RGB-based segmentation masks without relying on depth data.

## 8.6 Final Comparison: RGB-only vs All-cameras

In this section we will compare the best multi-frame projection (frames 50, 100, 250) against the two best camera-specific projection (RGB only and all cameras) to evaluate the trade-offs between temporal accumulation and multi-camera coverage

In [13]:
import pandas as pd
import numpy as np

# Build labeled visible pointclouds for any set of frames and cameras
def build_labeled_visible_rows(frame_list, camera_names, pose_cache=None, mask_cache=None):
    if mask_cache is None:
        mask_cache = {}

    labeled_frames = []
    for frame_idx in frame_list:
        if frame_idx not in mask_cache:
            mask_cache[frame_idx] = load_mask(masks_dir, frame_idx)
        rgb_mask = mask_cache[frame_idx]
        rgb_pose = get_pose_for_camera_frame(frame_idx, 'camera-rgb', pose_cache=pose_cache)

        for cam in camera_names:
            visible_df = build_visible_cloud_for_camera(frame_idx, cam, pose_cache=pose_cache)
            if visible_df is None or len(visible_df) == 0:
                continue

            xyz_world = visible_df[['px_world', 'py_world', 'pz_world']].to_numpy(dtype=np.float64)
            labels = np.zeros(len(xyz_world), dtype=int)

            for i in range(len(xyz_world)):
                pixel = project_world_to_rgb_image(xyz_world[i], rgb_pose, rgb_calib, T_Device_Camera)
                if pixel is not None:
                    u = int(round(float(pixel[0])))
                    v = int(round(float(pixel[1])))
                    if 0 <= v < rgb_mask.shape[0] and 0 <= u < rgb_mask.shape[1]:
                        labels[i] = int(rgb_mask[v, u])

            labeled_df = visible_df.copy()
            labeled_df['semantic_label'] = labels
            labeled_frames.append(labeled_df)

    if len(labeled_frames) == 0:
        return pd.DataFrame(columns=['point_id', 'px_world', 'py_world', 'pz_world', 'source_camera', 'source_frame_idx', 'semantic_label'])

    return pd.concat(labeled_frames, ignore_index=True)

In [14]:
from plyfile import PlyData, PlyElement

# Utility function to save a labeled pointcloud as a colored PLY file
def save_label_colored_ply(df, out_path):
    if df is None or len(df) == 0:
        return
    rows = []
    for _, row in df.iterrows():
        label = int(row.get('semantic_label', 0))
        color = save_label_to_color.get(label, [180, 180, 180])
        rows.append((row['px_world'], row['py_world'], row['pz_world'], color[0], color[1], color[2], label))
    np_rows = np.array(rows, dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'), ('red', 'u1'), ('green', 'u1'), ('blue', 'u1'), ('label', 'u1')])
    el = PlyElement.describe(np_rows, 'vertex')
    PlyData([el], text=True).write(out_path)

In [15]:
comparison_frames = [50, 100, 250]
comparison_camera_sets = {
    'rgb_only': ['camera-rgb'],
    'all_cameras': ['camera-rgb', 'camera-slam-left', 'camera-slam-right'],
}
comparison_out_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'pointclouds', 'final_comparison')
os.makedirs(comparison_out_dir, exist_ok=True)

print("Building RGB-only projection (frames 50, 100, 250)...")
rgb_only = build_labeled_visible_rows(comparison_frames, comparison_camera_sets['rgb_only'], pose_cache=pose_cache_mc, mask_cache=mask_cache)

print("Building All-cameras projection (frames 50, 100, 250)...")
all_cameras = build_labeled_visible_rows(comparison_frames, comparison_camera_sets['all_cameras'], pose_cache=pose_cache_mc, mask_cache=mask_cache)

print("Filtering and accumulating valid labels...")
rgb_only = rgb_only.sort_values('semantic_label', ascending=False).drop_duplicates(subset='point_id', keep='first').reset_index(drop=True)
all_cameras = all_cameras.sort_values('semantic_label', ascending=False).drop_duplicates(subset='point_id', keep='first').reset_index(drop=True)

rgb_path = os.path.join(comparison_out_dir, 'rgb_only_frames_50_100_250.ply')
all_path = os.path.join(comparison_out_dir, 'all_cameras_frames_50_100_250.ply')

print("Saving pointclouds to disk")
save_label_colored_ply(rgb_only, rgb_path)
save_label_colored_ply(all_cameras, all_path)

print('Saved RGB-only cloud:', rgb_path)
print('Saved all-cameras cloud:', all_path)
print('RGB-only points:', len(rgb_only))
print('All-cameras points:', len(all_cameras))

Building RGB-only projection (frames 50, 100, 250)...
Building All-cameras projection (frames 50, 100, 250)...


KeyboardInterrupt: 

## 8.7 Majority Voting System

The previous multi-frame accumulation relied on an inclusive logic. While this maximizes coverage, it suffers from the **Semantic Shadow (Tunnel Effect)** so background points physically hidden behind the target object in a specific frame inherit the object's label, creating 3D outliers. 

To solve this, we introduce a **Majority Voting System**. For each 3D point, we collect the semantic label assigned by every frame. The final label is the statistical mode (the most frequent vote). This will filter out perspective outliers while remaining robust to occasional 2D segmentation errors.

In [16]:
import numpy as np
import pandas as pd
from scipy import stats

def accumulate_labels_multi_frame_voting(points_df, trajectory_df, frame_indices, rgb_calib, T_Device_Camera, masks_dir, provider, mask_cache=None):
    """
    Projects and accumulates labels using Majority Voting to eliminate semantic shadows.
    It assigns the most frequent label across all provided frames.
    """
    if mask_cache is None:
        mask_cache = {}
        
    num_points = len(points_df)
    num_frames = len(frame_indices)
    
    # Matrix to store the vote of each frame [Num_Points x Num_Frames]
    votes_matrix = np.zeros((num_points, num_frames), dtype=int)
    
    for i, frame_idx in enumerate(frame_indices):
        if frame_idx not in mask_cache:
            mask_cache[frame_idx] = load_mask(masks_dir, frame_idx)
        mask = mask_cache[frame_idx]
        
        pose_row = get_pose_from_vrs_timestamp(provider, trajectory_df, frame_idx)
        labeled_tmp = label_pointcloud_at_pose(points_df, pose_row, rgb_calib, T_Device_Camera, mask)
        
        # Record the vote for the current frame
        votes_matrix[:, i] = labeled_tmp['semantic_label'].to_numpy()
        
    # Vectorized voting logic is compute the most frequent value along the frames axis
    mode_result, _ = stats.mode(votes_matrix, axis=1, keepdims=False)
    winning_labels = mode_result.astype(int)
        
    labeled_df = points_df.copy()
    labeled_df['semantic_label'] = winning_labels
    
    return labeled_df

### 8.7.1 Evaluating Multi-Frame Sets with Voting
We will now re-evaluate our 3, 5, and 7 frame sets using the voting logic. Our goal is to see if we can now safely increase the number of frames to gain density without simultaneously increasing the number of outliers.

In [17]:
import os

# Output directory for the voting results
voting_pcd_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'pointclouds', 'voting_comparison')
os.makedirs(voting_pcd_dir, exist_ok=True)

voting_pointclouds = {}

# Iterate over the previously defined frame sets
for frames in frame_sets: 
    dict_key = f"voting_frames_{'_'.join(map(str, frames))}"
    print(f"\nProcessing voting accumulation for {len(frames)} frames: {frames}")
    
    labeled_pc = accumulate_labels_multi_frame_voting(
        points_df, trajectory_df, frames, rgb_calib, T_Device_Camera, masks_dir, provider, mask_cache=mask_cache
    )
    voting_pointclouds[dict_key] = labeled_pc
    
    # Save the resulting pointcloud to disk
    out_path = os.path.join(voting_pcd_dir, f"pointcloud_{dict_key}.ply")
    save_label_colored_ply(labeled_pc, out_path)
    print(f"Completed voting accumulation for {dict_key} with {len(labeled_pc)} points.")


Processing voting accumulation for 3 frames: [50, 100, 250]


Projecting points: 100%|██████████| 239524/239524 [00:17<00:00, 13704.49it/s]


Completed voting accumulation for voting_frames_50_100_250 with 239524 points.

Processing voting accumulation for 5 frames: [50, 100, 250, 350, 480]


Projecting points: 100%|██████████| 239524/239524 [00:18<00:00, 13142.79it/s]


Completed voting accumulation for voting_frames_50_100_250_350_480 with 239524 points.

Processing voting accumulation for 7 frames: [50, 100, 250, 350, 480, 150, 200]


Projecting points: 100%|██████████| 239524/239524 [00:18<00:00, 13052.97it/s]


Completed voting accumulation for voting_frames_50_100_250_350_480_150_200 with 239524 points.


### 8.8 Final Comparison: Voting on RGB-only vs All-Cameras

Thanks to the voting system's ability to filter out noise, we have identified that accumulating 5 frames provides the optimal balance between geometric density and accuracy. We now adopt the 5-frames set as our new baseline.

In this final experiment, we apply the Majority Voting system across **multiple frames AND multiple cameras** simultaneously. Our goal is to verify if this statistical approach can successfully integrate the peripheral vision of the SLAM cameras while neutralizing their massive cross-camera occlusion errors.

In [18]:
# Select the optimal frame set identified in the previous step
best_voting_frames = [50, 100, 250, 350, 480]

print(f"Collecting raw projections for RGB-only ({len(best_voting_frames)} frames)...")
raw_rgb_votes = build_labeled_visible_rows(
    best_voting_frames, 
    comparison_camera_sets['rgb_only'], 
    pose_cache=pose_cache_mc, 
    mask_cache=mask_cache
)

print(f"Collecting raw projections for All-cameras ({len(best_voting_frames)} frames)...")
raw_all_votes = build_labeled_visible_rows(
    best_voting_frames, 
    comparison_camera_sets['all_cameras'], 
    pose_cache=pose_cache_mc, 
    mask_cache=mask_cache
)

In [19]:
def apply_voting(df):
    """
    Groups multiple observations of the same 3D point across different cameras/frames
    and elects the majority label to resolve spatial and temporal occlusions.
    """
    if df.empty: 
        return df
    
    # Find the majority vote for each unique point_id
    voted_labels = df.groupby('point_id')['semantic_label'].agg(lambda x: x.mode().iloc[0]).reset_index()
    
    # Extract the unique 3D geometry (keeping only one XYZ coordinate per point_id)
    geometry = df[['point_id', 'px_world', 'py_world', 'pz_world']].drop_duplicates('point_id')
    
    # Merge geometry with the winning votes
    final_df = geometry.merge(voted_labels, on='point_id')
    return final_df

print("Calculating Majority Vote across all views")
voted_rgb_only = apply_voting(raw_rgb_votes)
voted_all_cameras = apply_voting(raw_all_votes)

# Define paths for the ultimate comparison files
rgb_voting_path = os.path.join(voting_pcd_dir, 'ultimate_rgb_voting.ply')
all_voting_path = os.path.join(voting_pcd_dir, 'ultimate_all_cameras_voting.ply')

# Save pointclouds to disk
save_label_colored_ply(voted_rgb_only, rgb_voting_path)
save_label_colored_ply(voted_all_cameras, all_voting_path)

print(f'RGB-only: {len(voted_rgb_only)} points saved to {rgb_voting_path}')
print(f'All-cameras: {len(voted_all_cameras)} points saved to {all_voting_path}')

Calculating Majority Vote across all views
RGB-only: 181025 points saved to ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\voting_comparison\ultimate_rgb_voting.ply
All-cameras: 222760 points saved to ..\data\outputs\segmentation\kettle_and_forklift\pointclouds\voting_comparison\ultimate_all_cameras_voting.ply


### Final Conclusion

The transition from an inclusive logic to a Majority Voting system successfully neutralized the "semantic shadow" effect. 

By electing the statistical mode of the labels assigned across multiple views, we were able to safely scale up the temporal accumulation (using 5 frames instead of 3), capturing a much denser geometry without diluting the accuracy. Furthermore, applying this voting system to the multi-camera setup successfully integrated the peripheral vision of the SLAM cameras while heavily mitigating their cross-projection occlusion errors.